## Reading data for Spain

In [62]:
import pandas as pd

xls = pd.read_excel('../Extacted data/Pharm stats /Spain_atc3_2021_2025.xlsx', sheet_name=None)
print(xls.keys())

dict_keys(['ENVASES (miles)', 'PVP (miles EUR)'])


In [63]:
# have null value in column 'CHEMICAL_SUBSTANCE_BNF_DESCR'
df1 = xls['ENVASES (miles)']
df1.head()

,code,description_es,description_en,jan-2021,feb-2021,mar-2021,apr-2021,may-2021,jun-2021,jul-2021,...,mar-2025,apr-2025,may-2025,jun-2025,jul-2025,aug-2025,sep-2025,oct-2025,nov-2025,dec-2025
0,A01A,PREPARADOS ESTOMATOLÓGICOS,Stomatological preparations,13.17,14.93,17.25,15.26,16.31,17.03,16.63,...,19.43,18.89,19.42,16.84,11.49,6.66,5.33,3.32,1.56,0.79
1,A02B,AGENTES CONTRA LA ÚLCERA PÉPTICA Y EL REFLUJO ...,Drugs for peptic ulcer and gastro-oesophageal ...,5801.42,5557.14,6452.69,5904.73,5982.98,6064.39,6099.80,...,6065.69,5977.74,6059.48,5772.65,6149.48,5595.66,5980.83,6145.03,5651.30,6153.47
2,A03A,AGENTES CONTRA PADECIMIENTOS FUNCIONALES DEL E...,Drugs for functional gastrointestinal disorders,81.28,84.08,95.35,88.46,90.61,92.39,92.03,...,99.71,96.55,100.42,95.28,101.82,90.75,98.31,104.59,96.87,98.87
3,A03B,"BELLADONA Y DERIVADOS, MONOFÁRMACOS","Belladonna and derivatives, plain",57.66,58.65,64.81,61.24,62.78,63.71,65.82,...,81.30,77.08,80.86,77.28,85.33,82.01,83.62,89.40,83.18,84.00
4,A03F,PROPULSIVOS,Propulsives,447.53,436.87,501.41,464.51,467.64,498.33,506.23,...,555.97,530.22,538.84,516.49,554.29,503.15,535.35,584.23,544.21,574.69


## Fill in nulls in CHEMICAL_SUBSTANCE_BNF_DESCR column from BNF_CHEMICAL_SUBSTANCE

In [64]:
# filter only antidepressants
df_antidepressants = df1[df1["description_en"].isin(["Antidepressants", "Anxiolytics"])]

In [65]:
df_antidepressants

,code,description_es,description_en,jan-2021,feb-2021,mar-2021,apr-2021,may-2021,jun-2021,jul-2021,...,mar-2025,apr-2025,may-2025,jun-2025,jul-2025,aug-2025,sep-2025,oct-2025,nov-2025,dec-2025
132,N05B,ANSIOLÍTICOS,Anxiolytics,4802.61,4619.32,5296.63,4896.67,4917.09,4952.42,4955.85,...,4589.21,4490.54,4568.76,4325.21,4564.88,4179.57,4434.95,4564.21,4157.13,4449.77
134,N06A,ANTIDEPRESIVOS,Antidepressants,3892.41,3720.43,4335.07,4022.20,4082.91,4158.32,4211.59,...,5018.19,4960.36,5068.57,4840.89,5197.63,4723.19,5056.89,5219.98,4822.68,5213.79


In [66]:
df_long = df_antidepressants.melt(
    id_vars=["code", "description_es", "description_en"],
    var_name="date",
    value_name="items_1000"
)
df_long.head()

,code,description_es,description_en,date,items_1000
0,N05B,ANSIOLÍTICOS,Anxiolytics,jan-2021,4802.61
1,N06A,ANTIDEPRESIVOS,Antidepressants,jan-2021,3892.41
2,N05B,ANSIOLÍTICOS,Anxiolytics,feb-2021,4619.32
3,N06A,ANTIDEPRESIVOS,Antidepressants,feb-2021,3720.43
4,N05B,ANSIOLÍTICOS,Anxiolytics,mar-2021,5296.63


## Dropping columns 

In [67]:
df = df_long.drop(columns=['description_es','code'])

In [68]:
df = df.rename(columns={'description_en': 'group',})
df = df[["date", "group", "items_1000"]]
df.head(2)

,date,group,items_1000
0,jan-2021,Anxiolytics,4802.61
1,jan-2021,Antidepressants,3892.41


In [69]:
df['date'] = pd.to_datetime(df['date'], format='%b-%Y').dt.strftime('%Y-%m')

In [81]:
df['country']= 'Spain' 
df['items'] = df['items_1000']*1000
df.head()

,date,country,group,items_1000,items
0,2021-01,Spain,Anxiolytics,4802.61,4802610.0
1,2021-01,Spain,Antidepressants,3892.41,3892410.0
2,2021-02,Spain,Anxiolytics,4619.32,4619320.0
3,2021-02,Spain,Antidepressants,3720.43,3720430.0
4,2021-03,Spain,Anxiolytics,5296.63,5296630.0


In [82]:
df = df[["date","country","group","items_1000",'items']]
df.head()

,date,country,group,items_1000,items
0,2021-01,Spain,Anxiolytics,4802.61,4802610.0
1,2021-01,Spain,Antidepressants,3892.41,3892410.0
2,2021-02,Spain,Anxiolytics,4619.32,4619320.0
3,2021-02,Spain,Antidepressants,3720.43,3720430.0
4,2021-03,Spain,Anxiolytics,5296.63,5296630.0


In [83]:
# save to csv
df.to_csv("Spain_antidepressants_and_anxiolytics_21_25.csv", index=False)

In [74]:
# convert to monthly period
df["date"] = pd.PeriodIndex(df["date"], freq="M")

# filter range
df_group = df[
    (df["date"] >= "2021-01") &
    (df["date"] <= "2025-12")
]

# group and sum
result = (
    df_group
    .groupby(["date"])["items_1000"]
    .sum()
    .reset_index()
)

print(result)

       date  items_1000
0   2021-01     8695.02
1   2021-02     8339.75
2   2021-03     9631.70
3   2021-04     8918.87
4   2021-05     9000.00
5   2021-06     9110.74
6   2021-07     9167.44
7   2021-08     8857.93
8   2021-09     9072.62
9   2021-10     9076.12
10  2021-11     9274.37
11  2021-12     9385.86
12  2022-01     9118.50
13  2022-02     8592.18
14  2022-03     9979.73
15  2022-04     9131.59
16  2022-05     9440.06
17  2022-06     9264.30
18  2022-07     9043.67
19  2022-08     9276.02
20  2022-09     9282.94
21  2022-10     9285.34
22  2022-11     9341.64
23  2022-12     9427.23
24  2023-01     9437.29
25  2023-02     8667.03
26  2023-03     9898.80
27  2023-04     8899.12
28  2023-05     9808.99
29  2023-06     9423.92
30  2023-07     9302.79
31  2023-08     9333.27
32  2023-09     9220.36
33  2023-10     9597.03
34  2023-11     9397.14
35  2023-12     9163.96
36  2024-01     9788.53
37  2024-02     9135.67
38  2024-03     9090.52
39  2024-04     9918.88
40  2024-05     